<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/22-research-practice-emerging-frontiers.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Research Practice and Emerging Frontiers** {#research-practice-emerging-frontiers}

Deep learning research is not the production of a larger model followed by a favorable score. It is a disciplined process for turning an important uncertainty into a falsifiable question, constructing evidence that separates plausible explanations, and stating exactly how far the conclusion travels. The research artifact therefore includes the data lineage, code, environment, rejected hypotheses, resource budget, and failure analysis, not only the final checkpoint.

The first half of this chapter builds that process. The second half uses it to examine emerging directions without treating every recent result as a durable paradigm. A direction is described as **established** when mechanisms and evaluation protocols have accumulated across independent work, **active** when promising results coexist with major unresolved constraints, and **speculative** when the central assumptions or deployment path remain weakly tested.

The code follows one compact research question on scikit-learn's copy of the [UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49) dataset, released under **CC BY 4.0**: *under a fixed training budget, how do width and dropout affect clean accuracy, corrupted-input accuracy, and uncertainty?* A deterministic train/validation/test split is reused for baselines, ablations, scaling pilots, sparse computation, test-time compute, continual learning, causal-shift demonstrations, and constrained decoding. These experiments teach research mechanics; an 8-by-8 digit benchmark is not evidence for frontier-scale behavior.

![The research cycle connects planning, collection, processing, analysis, publication, preservation, and reuse.](assets/dl22-research-cycle.jpg){fig-align="center" width="76%" fig-alt="A hand-drawn circular research process connects data planning, collection, processing, analysis, publishing, preserving, and reuse."}

*Source: The Turing Way Community and Scriberia, [Research Cycle](https://doi.org/10.5281/zenodo.3332807), CC BY 4.0. The cycle emphasizes that publication is not the endpoint: preserved artifacts enable scrutiny and reuse.*

<details>
<summary><strong>PyTorch: establish the shared research workload</strong></summary>

```python
import hashlib
import json
import platform
import random
import time

import numpy as np
import sklearn
import torch
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=2201):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_ids = np.arange(len(digits.data))
train_ids, heldout_ids = train_test_split(
    all_ids, test_size=0.40, stratify=digits.target, random_state=2201
)
val_ids, test_ids = train_test_split(
    heldout_ids,
    test_size=0.50,
    stratify=digits.target[heldout_ids],
    random_state=2201,
)

# UCI documents pixel intensities on a fixed 0..16 scale, so this transform
# does not estimate a statistic from validation or test examples.
images = torch.tensor(digits.images / 16.0, dtype=torch.float32).flatten(1)
targets = torch.tensor(digits.target, dtype=torch.long)


def make_loader(ids, batch_size=64, shuffle=False, seed=2201):
    dataset = TensorDataset(images[ids], targets[ids])
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)


class DigitMLP(nn.Module):
    def __init__(self, width=64, dropout=0.10):
        super().__init__()
        self.width = width
        self.fc1 = nn.Linear(64, width)
        self.fc2 = nn.Linear(width, width)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(width, 10)

    def features(self, x):
        h1 = F.gelu(self.fc1(x))
        h2 = F.gelu(self.fc2(self.dropout(h1)))
        return h1, h2

    def forward(self, x):
        return self.out(self.features(x)[1])


def evaluate(model, ids, noise_std=0.0, seed=0):
    model.eval()
    x = images[ids]
    if noise_std:
        generator = torch.Generator().manual_seed(seed)
        noise = torch.randn(x.shape, generator=generator) * noise_std
        x = (x + noise).clamp(0.0, 1.0)
    with torch.inference_mode():
        logits = model(x)
        probabilities = logits.softmax(dim=1)
    accuracy = float(logits.argmax(1).eq(targets[ids]).float().mean())
    nll = float(F.cross_entropy(logits, targets[ids]))
    return {"accuracy": accuracy, "nll": nll, "probabilities": probabilities, "logits": logits}


def fit_model(seed=2201, width=64, dropout=0.10, epochs=12, lr=2e-3):
    seed_everything(seed)
    model = DigitMLP(width=width, dropout=dropout)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loader = make_loader(train_ids, shuffle=True, seed=seed)
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            loss = F.cross_entropy(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    clean = evaluate(model, test_ids)
    corrupt = evaluate(model, test_ids, noise_std=0.22, seed=999)
    return model, {
        "clean_accuracy": clean["accuracy"],
        "corrupt_accuracy": corrupt["accuracy"],
        "clean_nll": clean["nll"],
    }


baseline_model, baseline_metrics = fit_model()
assert len(set(train_ids) & set(test_ids)) == 0
assert baseline_metrics["clean_accuracy"] > 0.88
print({k: round(v, 3) for k, v in baseline_metrics.items()})
```

</details>

The split unit is an image because UCI Digits has no subject identifier in this packaged copy. In a medical, user, speaker, household, or time-dependent study, splitting individual rows would often leak identity or future information. That boundary must be decided from the data-generating process, not from whichever split call is convenient.


### **Reading and Mapping a Research Paper** {#reading-mapping-research-paper}

A paper should first be read as an argument. The abstract advertises a result, but the scientific object is a chain linking a question, a proposed mechanism, evidence, assumptions, and a bounded contribution. Reading only in document order makes it easy to remember architecture details while missing whether the comparison answers the stated question.

![A research paper can be mapped as a chain from question to bounded contribution.](assets/dl22-paper-claim-map.svg){fig-align="center" width="76%" fig-alt="Five boxes connect research question, mechanism, evidence, boundary, and contribution."}

Start with a **claim map**. Write the primary claim in one sentence with its population, intervention, comparator, metric, and budget. Then identify the mechanism that is supposed to cause the gain. For every result table, ask which alternative explanation it rules out: more parameters, more data, a different split, longer tuning, favorable preprocessing, or selective reporting. Finally, extract the conditions under which the claim is not tested. A result on one benchmark and one seed is an observation; it is not yet a general law.

Three passes are usually more efficient than line-by-line reading:

1. **Triage:** inspect the question, result tables, limitations, data, and compute. Decide whether the paper is relevant and whether the evidence can support the advertised scope.
2. **Mechanism:** trace tensor shapes, objectives, algorithm steps, and computational cost. Re-derive the smallest equation that distinguishes the method from its baseline.
3. **Audit:** reconstruct the split, tuning budget, uncertainty, ablations, release artifacts, and possible confounders. The [NeurIPS paper checklist](https://neurips.cc/public/guides/PaperChecklist) is useful here because it turns reporting expectations into concrete questions about assumptions, reproducibility, resources, and societal impact.

<details>
<summary><strong>Python: encode a claim map that can fail validation</strong></summary>

```python
claim_card = {
    "question": "Do width and dropout improve corrupted-digit accuracy at fixed epochs?",
    "population": "UCI Digits under synthetic Gaussian corruption",
    "intervention": {"width": [64, 128], "dropout": [0.0, 0.2]},
    "comparator": "width=64, dropout=0.0",
    "primary_metric": "corrupt_accuracy",
    "secondary_metrics": ["clean_accuracy", "clean_nll"],
    "fixed_budget": {"epochs": 8, "seeds": [2201, 2202, 2203]},
    "boundary": "mechanism study on 8x8 digits, not a scale-law benchmark",
}

required = {
    "question",
    "population",
    "intervention",
    "comparator",
    "primary_metric",
    "fixed_budget",
    "boundary",
}
assert required <= claim_card.keys()
assert claim_card["primary_metric"] not in claim_card["secondary_metrics"]
print(json.dumps(claim_card, indent=2))
```

</details>

The card is intentionally stricter than a prose summary. If the comparator, budget, or population cannot be filled in, the reader has found an ambiguity that matters to interpretation or reproduction. Citation count and benchmark rank do not repair such an ambiguity.


### **Reproduction, Replication, and Baselines** {#reproduction-replication-baselines}

Terminology varies across fields, so a project should define it operationally. Here, **repeatability** means rerunning the same code and environment; **computational reproduction** means regenerating the reported result from released artifacts; **replication** means testing the same scientific claim with an independently constructed implementation or dataset; and **robustness analysis** means changing plausible nuisance conditions such as seeds, splits, budgets, or domains. A deterministic rerun is valuable, but it tests a much narrower proposition than independent replication.

![Repeatability, reproduction, replication, robustness, and decision scope form progressively stronger evidence.](assets/dl22-reproducibility-chain.svg){fig-align="center" width="76%" fig-alt="A five-stage evidence chain progresses from repeatability to a bounded decision, with a baseline ladder below."}

A baseline is a control, not a ceremonial weak model. A useful ladder includes a sanity baseline that detects leakage, a simple method that quantifies task difficulty, a strong method matched for data and tuning budget, and an upper bound only when the extra information is legitimate. Comparisons must hold constant the resources that are not part of the research question. If a new method receives more tokens, augmentations, hyperparameter trials, or test-time calls, those are interventions and must be reported.

Reproducibility requires identifying the complete state that affects the outcome: dataset version and checksum, split indices, preprocessing, package versions, random seeds, model configuration, optimizer, number of updates, hardware-sensitive kernels, and evaluation code. Bitwise identity is neither always possible nor always scientifically necessary; the target may instead be agreement within a declared numerical or statistical tolerance.

<details>
<summary><strong>Python: verify deterministic reruns and create a minimal manifest</strong></summary>

```python
def state_digest(model):
    payload = b"".join(
        tensor.detach().cpu().contiguous().numpy().tobytes()
        for tensor in model.state_dict().values()
    )
    return hashlib.sha256(payload).hexdigest()


repeat_a, metrics_a = fit_model(seed=2207, epochs=6)
repeat_b, metrics_b = fit_model(seed=2207, epochs=6)
assert state_digest(repeat_a) == state_digest(repeat_b)
assert metrics_a == metrics_b

dataset_digest = hashlib.sha256(
    np.ascontiguousarray(digits.data).tobytes()
    + np.ascontiguousarray(digits.target).tobytes()
).hexdigest()
manifest = {
    "dataset": "UCI Optical Recognition of Handwritten Digits",
    "dataset_sha256": dataset_digest,
    "split_sha256": hashlib.sha256(
        np.concatenate([train_ids, val_ids, test_ids]).astype(np.int64).tobytes()
    ).hexdigest(),
    "seed": 2207,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "sklearn": sklearn.__version__,
    "model_sha256": state_digest(repeat_a),
}
assert len(manifest["dataset_sha256"]) == 64
print({"repeatable": True, "clean_accuracy": round(metrics_a["clean_accuracy"], 3)})
```

</details>

This test establishes repeatability on the present CPU software stack. It does not establish replication, and it should not be generalized to different accelerator kernels. A release should state which level it promises and provide the command that regenerates every reported table or figure.


### **Ablations and Controlled Experiments** {#ablations-controlled-experiments}

An ablation asks which component is responsible for an observed difference. Removing a component from the final model is informative only if the comparison preserves training budget, data, parameter accounting, tuning policy, and evaluation. Otherwise the experiment changes several causes at once.

For two binary factors, width (A) and dropout (B), a full (2\times2) design estimates both main effects and their interaction. If (m_{ab}) is the mean metric at factor levels (a,b\in\{0,1\}), the interaction is

$$
I_{AB}=(m_{11}-m_{10})-(m_{01}-m_{00}).
$$

A nonzero (I_{AB}) means the effect of dropout depends on width. Reporting only the reference and the combined model cannot distinguish two additive benefits from an interaction or from one harmful component masked by another.

![A two-by-two ablation identifies main effects and interaction.](assets/dl22-ablation-matrix.svg){fig-align="center" width="72%" fig-alt="A two by two experimental matrix crosses hidden width and dropout and shows the interaction contrast."}

<details>
<summary><strong>PyTorch: run a multi-seed factorial ablation</strong></summary>

```python
ablation_rows = []
ablation_models = {}
for width in (64, 128):
    for dropout in (0.0, 0.2):
        for seed in (2201, 2202, 2203):
            candidate, metrics = fit_model(
                seed=seed, width=width, dropout=dropout, epochs=8
            )
            ablation_models[(width, dropout, seed)] = candidate
            ablation_rows.append(
                {"width": width, "dropout": dropout, "seed": seed, **metrics}
            )


def grouped_mean(metric):
    return {
        (width, dropout): float(
            np.mean(
                [
                    row[metric]
                    for row in ablation_rows
                    if row["width"] == width and row["dropout"] == dropout
                ]
            )
        )
        for width in (64, 128)
        for dropout in (0.0, 0.2)
    }


corrupt_means = grouped_mean("corrupt_accuracy")
interaction = (
    corrupt_means[(128, 0.2)]
    - corrupt_means[(128, 0.0)]
    - corrupt_means[(64, 0.2)]
    + corrupt_means[(64, 0.0)]
)
assert len(ablation_rows) == 12
print({
    "means": {str(k): round(v, 3) for k, v in corrupt_means.items()},
    "interaction": round(interaction, 3),
})
```

</details>

Three seeds provide a variance warning, not a precise population estimate. Report individual runs, aggregation, and uncertainty rather than selecting the best seed. If many variants are inspected, the validation set becomes part of the optimization loop; the final test must remain untouched until the decision rule is frozen. Negative ablations belong in the research log because they constrain future explanations even when they do not improve the headline score.


### **Scaling Laws and Compute Allocation** {#scaling-laws-compute-allocation}

Scaling laws are empirical regularities connecting loss to resources such as parameter count (N), dataset size (D), and training compute (C). A common local model is

$$
L(C)=L_{\infty}+A C^{-\alpha},
$$

where (L_{\infty}) is an irreducible or asymptotic floor, (A>0) sets the reducible-loss scale, and (alpha>0) is the observed slope on log-log axes after subtracting the floor. The equation is not a theorem about all architectures or data. It is a fitted description over a measured regime.

[Kaplan et al.](https://arxiv.org/abs/2001.08361) documented power-law trends across language-model scale, while [Hoffmann et al.](https://arxiv.org/abs/2203.15556) showed that model size and training tokens must be allocated jointly under fixed compute. The broader lesson is experimental: a parameter-only comparison is incomplete, and a scaling pilot should vary multiple resource axes before committing a full budget.

![Measured pilots support local interpolation more strongly than distant extrapolation.](assets/dl22-scaling-curve.svg){fig-align="center" width="74%" fig-alt="A decreasing loss curve shows measured pilot points and a dashed uncertain extrapolation region."}

<details>
<summary><strong>PyTorch: build a small scaling pilot without claiming a universal law</strong></summary>

```python
scaling_rows = []
for width in (16, 32, 64, 128):
    pilot_model, pilot_metrics = fit_model(
        seed=2210, width=width, dropout=0.1, epochs=7
    )
    parameters = sum(p.numel() for p in pilot_model.parameters())
    compute_proxy = parameters * len(train_ids) * 7
    scaling_rows.append({
        "width": width,
        "parameters": parameters,
        "compute_proxy": compute_proxy,
        "validation_nll": evaluate(pilot_model, val_ids)["nll"],
    })

# A two-parameter log-log fit is a diagnostic summary, not a forecast guarantee.
log_c = np.log([row["compute_proxy"] for row in scaling_rows])
log_l = np.log([row["validation_nll"] for row in scaling_rows])
slope, intercept = np.polyfit(log_c, log_l, deg=1)
assert all(np.isfinite([slope, intercept]))
print({"local_exponent": round(-float(slope), 3), "pilots": scaling_rows})
```

</details>

This proxy ignores backward-pass constants, memory bandwidth, kernel efficiency, and hyperparameter retuning. A serious scaling study records tokens or examples seen, FLOPs or accelerator-hours, wall-clock time, energy when relevant, and inference cost. Extrapolation should include uncertainty and a stop rule: if a pilot violates the assumed monotonic regime or the projected gain is smaller than measurement noise, more scale is not yet justified.


### **Sparse and Adaptive Computation** {#sparse-adaptive-computation}

Dense networks activate most parameters for every example. Sparse computation attempts to spend capacity selectively. **Weight sparsity** removes individual connections, **structured sparsity** removes blocks, channels, heads, or layers, and **conditional computation** chooses different paths for different inputs. These forms have different hardware consequences even when they contain the same number of zeros.

For hidden activations (h\in\mathbb{R}^{D}), a top-(k) gate keeps the index set

$$
S_k(h)=\operatorname{TopK}(|h|,k),\qquad
\tilde h_i=h_i\,\mathbb{1}[i\in S_k(h)].
$$

This reduces active values from (D) to (k), but a dense implementation may still compute all (D) activations before masking. Actual speed requires kernels and storage layouts that avoid the skipped work.

![Conditional sparsity routes an input through only a subset of paths.](assets/dl22-sparse-adaptive-computation.svg){fig-align="center" width="75%" fig-alt="An input enters a top-k gate, activates two computation paths, leaves one inactive, and merges the result."}

<details>
<summary><strong>PyTorch: isolate the accuracy effect of top-k hidden activations</strong></summary>

```python
def logits_with_topk(model, x, k):
    model.eval()
    with torch.inference_mode():
        h1 = F.gelu(model.fc1(x))
        indices = h1.abs().topk(k, dim=1).indices
        mask = torch.zeros_like(h1).scatter_(1, indices, 1.0)
        sparse_h1 = h1 * mask
        h2 = F.gelu(model.fc2(sparse_h1))
        return model.out(h2), float(mask.mean())


sparse_results = {}
for k in (8, 16, 32, baseline_model.width):
    logits, active_fraction = logits_with_topk(baseline_model, images[test_ids], k)
    sparse_results[k] = {
        "active_fraction": active_fraction,
        "accuracy": float(logits.argmax(1).eq(targets[test_ids]).float().mean()),
    }

dense_logits = evaluate(baseline_model, test_ids)["logits"]
full_topk_logits, _ = logits_with_topk(
    baseline_model, images[test_ids], baseline_model.width
)
assert torch.allclose(dense_logits, full_topk_logits, atol=1e-6)
print({k: {m: round(v, 3) for m, v in row.items()} for k, row in sparse_results.items()})
```

</details>

The experiment is a post-training activation intervention, not sparse training and not a latency benchmark. It answers a narrow diagnostic question: how much of this model's hidden activity can be removed before predictions change? Deployment research must separately measure end-to-end latency, memory traffic, batching behavior, compilation support, and the cost of deciding what to skip.


### **Mixture-of-Experts and Hybrid Architectures** {#mixture-experts-hybrid-architectures}

A mixture-of-experts (MoE) layer contains several parameterized experts but activates only a small subset for each token or example. Given router probabilities (g_i(x)) and expert outputs (E_i(x)), a top-(k) layer computes

$$
y(x)=\sum_{i\in\operatorname{TopK}(g(x),k)}\hat g_i(x)E_i(x),
$$

where (hat g_i) renormalizes the selected weights. Total parameters may grow with the number of experts while active FLOPs remain closer to a dense layer with only (k) experts. The benefit is conditional capacity; the price is routing complexity, expert communication, capacity overflow, and instability when many inputs choose the same expert.

[Switch Transformer](https://www.jmlr.org/papers/v23/21-0998.html) demonstrated simplified top-1 routing at large scale and documented communication and stability constraints. Hybrid architectures combine mechanisms with complementary properties: [Jamba](https://arxiv.org/abs/2403.19887), for example, interleaves attention, state-space, and MoE components. Such systems should be compared on quality, active FLOPs, memory footprint, communication, throughput, and long-context behavior rather than parameter count alone.

![A hybrid block can combine attention, state-space recurrence, and sparse experts.](assets/dl22-moe-hybrid.svg){fig-align="center" width="78%" fig-alt="Tokens pass through attention, a state-space block, and a routed mixture-of-experts layer."}

<details>
<summary><strong>PyTorch: train a tiny top-1 MoE and inspect routing balance</strong></summary>

```python
class TinyMoE(nn.Module):
    def __init__(self, width=48, experts=4):
        super().__init__()
        self.trunk = nn.Linear(64, width)
        self.router = nn.Linear(width, experts)
        self.experts = nn.ModuleList([nn.Linear(width, width) for _ in range(experts)])
        self.out = nn.Linear(width, 10)

    def forward(self, x):
        h = F.gelu(self.trunk(x))
        router_probs = self.router(h).softmax(dim=1)
        choices = router_probs.argmax(dim=1)
        all_outputs = torch.stack([F.gelu(expert(h)) for expert in self.experts], dim=1)
        chosen = all_outputs[torch.arange(len(x)), choices]
        return self.out(chosen), router_probs, choices


seed_everything(2220)
moe_model = TinyMoE()
moe_optimizer = torch.optim.AdamW(moe_model.parameters(), lr=2e-3)
for _ in range(9):
    moe_model.train()
    for x, y in make_loader(train_ids, shuffle=True, seed=2220):
        logits, router_probs, _ = moe_model(x)
        mean_load = router_probs.mean(dim=0)
        balance_loss = ((mean_load - 0.25) ** 2).mean()
        loss = F.cross_entropy(logits, y) + 0.2 * balance_loss
        moe_optimizer.zero_grad()
        loss.backward()
        moe_optimizer.step()

moe_model.eval()
with torch.inference_mode():
    moe_logits, _, moe_choices = moe_model(images[test_ids])
route_counts = torch.bincount(moe_choices, minlength=4)
moe_accuracy = float(moe_logits.argmax(1).eq(targets[test_ids]).float().mean())
assert int(route_counts.sum()) == len(test_ids)
print({"accuracy": round(moe_accuracy, 3), "routes": route_counts.tolist()})
```

</details>

The auxiliary loss encourages soft router probabilities to spread, but balanced averages do not guarantee meaningful specialization. Diagnose token drops, per-domain routing, router entropy, expert gradients, and all-to-all communication. An MoE can have excellent theoretical active-FLOP efficiency and poor wall-clock utilization if batches are small or routes are imbalanced.


### **Test-Time Computation** {#test-time-computation}

Test-time computation trades additional inference work for a better decision. General forms include ensembling transformations, drawing several candidate outputs, verifying and selecting candidates, iteratively revising a state, or searching a tree of actions. In reasoning systems, the compute allocation itself becomes a policy: easy inputs should stop early while difficult inputs receive more samples or deeper search.

Let (q(y\mid x,c)) denote solution quality after compute budget (c). The practical objective is not simply to maximize quality but to choose

$$
c^*(x)=\arg\max_c\;\mathbb{E}[U(y,x)\mid c]-\lambda\,\operatorname{Cost}(c),
$$

where (U) is task utility and (lambda) converts latency, money, or energy into the same decision scale. [Snell et al.](https://arxiv.org/abs/2408.03314) found that effective allocation depends on problem difficulty and the base model; this is evidence against a universal rule such as “always sample more.”

![A controller can choose a fast pass, repeated sampling, or deeper search.](assets/dl22-test-time-compute.svg){fig-align="center" width="76%" fig-alt="An adaptive controller routes an input to one-pass, multi-sample, or search-based inference before producing an answer."}

<details>
<summary><strong>PyTorch: measure a Monte Carlo dropout compute ladder</strong></summary>

```python
corrupt_x = images[test_ids].clone()
generator = torch.Generator().manual_seed(2230)
corrupt_x = (corrupt_x + 0.22 * torch.randn(corrupt_x.shape, generator=generator)).clamp(0, 1)


def mc_dropout_predict(model, x, passes):
    model.train()  # Dropout remains stochastic; this network has no batch normalization.
    samples = []
    with torch.inference_mode():
        for _ in range(passes):
            samples.append(model(x).softmax(dim=1))
    mean_probability = torch.stack(samples).mean(dim=0)
    entropy = -(mean_probability * mean_probability.clamp_min(1e-9).log()).sum(dim=1)
    return mean_probability, entropy


test_time_rows = []
for passes in (1, 4, 16):
    probability, entropy = mc_dropout_predict(baseline_model, corrupt_x, passes)
    accuracy = float(probability.argmax(1).eq(targets[test_ids]).float().mean())
    test_time_rows.append({
        "passes": passes,
        "accuracy": accuracy,
        "mean_entropy": float(entropy.mean()),
    })
baseline_model.eval()
assert [row["passes"] for row in test_time_rows] == [1, 4, 16]
print([{k: round(v, 3) if isinstance(v, float) else v for k, v in row.items()} for row in test_time_rows])
```

</details>

Extra passes may reduce sampling noise but need not improve accuracy monotonically. A valid study reports the entire quality-cost curve, includes verifier or controller cost, and compares against spending the same budget on a stronger single-pass model. Search also amplifies evaluator weaknesses: optimizing against an imperfect verifier can select fluent but incorrect candidates.


### **Embodied Learning and World Models** {#embodied-learning-world-models}

Embodied learning places an agent inside a feedback loop. The agent receives partial observations (o_t), chooses actions (a_t), changes the environment, and receives future observations and rewards. Unlike a fixed supervised dataset, the policy influences which data will exist. Exploration, delayed consequences, safety constraints, and distribution shift are therefore central parts of the learning problem.

A world model compresses interaction history into a latent state (z_t), predicts consequences, and supports learning or planning in imagined trajectories:

$$
z_t\sim q_\phi(z_t\mid z_{t-1},a_{t-1},o_t),\qquad
(z_{t+1},r_t,\gamma_t)\sim p_\theta(\cdot\mid z_t,a_t).
$$

Here (q_\phi) is an inference model, (p_\theta) predicts latent dynamics, reward, and continuation (gamma_t). The policy can improve from simulated rollouts, but only within the model's learned support.

![A world model closes a loop between observation, latent dynamics, imagined rollouts, actions, and reality checks.](assets/dl22-world-model-loop.svg){fig-align="center" width="73%" fig-alt="An observation is encoded into latent state, rolled through learned dynamics, evaluated by a policy, acted in reality, and checked for model error."}

[DreamerV3](https://www.nature.com/articles/s41586-025-08744-2) is an established example of learning behavior through latent imagination across diverse control domains. The frontier is broader: models that learn from video and interaction, plan over long horizons, transfer across bodies and environments, and combine generative prediction with reliable control. The unresolved issues are not merely visual realism. A useful world model must represent action-conditioned causality, uncertainty, rare hazards, object persistence, and the consequences of interventions.

Research claims should separate **prediction quality**, **planning utility**, and **real-world validity**. Low reconstruction error can ignore decision-critical details; visually plausible rollouts can violate dynamics; an agent can exploit model errors. Evaluation therefore needs held-out interventions, long-horizon compounding-error tests, uncertainty under novelty, and safety checks before imagined policies act in a physical system. Chapter 17 develops the core reinforcement-learning and world-model algorithms; this section frames their open research boundary.


### **Deep Learning for Science** {#deep-learning-for-science}

Scientific deep learning uses neural models as instruments inside a scientific workflow. Common roles include surrogate simulation, inverse problems, parameter estimation, experimental design, structure prediction, control, and hypothesis generation. Success is not defined by benchmark error alone. Predictions must respect units, symmetries, boundary conditions, conservation laws, uncertainty, and the domain's actual decision process.

![Scientific deep learning closes a loop from question and domain knowledge through a learned component to domain validation.](assets/dl22-scientific-learning-stack.svg){fig-align="center" width="72%" fig-alt="Four stacked layers connect a scientific question, data and governing knowledge, a learned component, and domain validation, with feedback to the question."}

[AlphaFold2](https://www.nature.com/articles/s41586-021-03819-2) demonstrated that carefully designed neural representations and domain evaluation can transform protein-structure prediction. [GraphCast](https://doi.org/10.1126/science.adi2336) learns global weather dynamics from reanalysis data, while [NeuralGCM](https://www.nature.com/articles/s41586-024-07744-y) combines differentiable atmospheric dynamics with learned components. These examples represent distinct patterns: data-driven prediction, learned simulators, and hybrid physics-neural systems. None implies that equations, experiments, or expert validation have become unnecessary.

Three validation layers are essential:

| Layer | Question | Typical evidence |
|---|---|---|
| Numerical | Does the model approximate the target on measured conditions? | held-out error, calibration, conservation residuals |
| Scientific | Does it preserve the mechanisms and invariances required by the domain? | intervention tests, dimensional checks, known limiting cases |
| Operational | Does it improve a real workflow under cost and risk constraints? | prospective studies, expert comparison, decision outcomes |

The largest risk is shortcut discovery from observational data. A model may exploit simulator artifacts, data-assimilation conventions, experimental batches, or geography. Scientific claims need external validation across instruments, institutions, regimes, or time, and should report where the model extrapolates beyond support. Hybrid models can encode trusted structure, but an incorrect constraint can be more damaging than an unconstrained approximation because it creates false confidence.


### **Neural Operators** {#neural-operators}

An ordinary neural network approximates a mapping between finite-dimensional vectors. A neural operator aims to learn a mapping between functions, such as a coefficient field (a(x)) and the corresponding PDE solution (u(x)):

$$
\mathcal{G}_\theta:a(x)\mapsto u(x).
$$

The [Fourier Neural Operator](https://arxiv.org/abs/2010.08895) parameterizes global mixing in frequency space. A simplified layer is

$$
v_{t+1}(x)=\sigma\!\left(Wv_t(x)+\mathcal{F}^{-1}\!\left(R_\theta(k)\,\mathcal{F}(v_t)(k)\right)(x)\right),
$$

where (W) is a local channel transform, (mathcal{F}) is the Fourier transform, and (R_\theta(k)) is a learned transform retained for selected modes. Because learned parameters act on modes rather than a fixed dense grid, the same operator can be evaluated at different resolutions when the discretization and representation support it.

![A Fourier neural operator transforms a field to frequency space, learns selected modes, and returns to the spatial grid.](assets/dl22-neural-operator.svg){fig-align="center" width="77%" fig-alt="A field passes through FFT, learned retained Fourier modes, inverse FFT, and a local residual transform."}

<details>
<summary><strong>PyTorch: verify the shape and shift behavior of a spectral operator</strong></summary>

```python
def low_mode_operator(x, modes=3):
    # x has shape [B, H, W]; this fixed filter demonstrates the spectral path.
    spectrum = torch.fft.rfft2(x)
    filtered = torch.zeros_like(spectrum)
    filtered[:, :modes, :modes] = spectrum[:, :modes, :modes]
    filtered[:, -modes + 1 :, :modes] = spectrum[:, -modes + 1 :, :modes]
    return torch.fft.irfft2(filtered, s=x.shape[-2:])


digit_fields = images[test_ids[:16]].reshape(-1, 8, 8)
filtered_fields = low_mode_operator(digit_fields, modes=3)
shifted = torch.roll(digit_fields, shifts=(1, 2), dims=(-2, -1))
filtered_shifted = low_mode_operator(shifted, modes=3)
expected_shift = torch.roll(filtered_fields, shifts=(1, 2), dims=(-2, -1))
assert filtered_fields.shape == digit_fields.shape
assert torch.allclose(filtered_shifted, expected_shift, atol=1e-5)
print({"shape": tuple(filtered_fields.shape), "relative_error": float((filtered_fields-digit_fields).norm()/digit_fields.norm())})
```

</details>

This code is a fixed low-pass operator on digit images, not a trained PDE solver. It demonstrates tensor flow and circular shift equivariance. Research-grade neural operators must test boundary conditions, aliasing, mesh geometry, resolution transfer, long rollout stability, conservation, and comparison with numerical solvers at matched error and compute. Zero-shot super-resolution is a hypothesis to evaluate for each problem, not an automatic consequence of using an FFT.


### **Continual and Lifelong Learning** {#continual-lifelong-learning}

Continual learning studies systems that update from a stream of tasks or domains without retraining from all historical data. The core tension is **plasticity** versus **stability**: parameters must absorb new information while preserving behavior that remains useful. Catastrophic forgetting occurs when updates for current data overwrite representations needed for earlier data.

Settings must be named precisely. In **task-incremental** learning, task identity is available at inference; in **domain-incremental** learning, the prediction space stays fixed while the input distribution changes; in **class-incremental** learning, new classes arrive and the model must classify among all classes without a task label. These settings have different difficulty and memory assumptions.

![A continual learner must adapt to new tasks while retaining performance on earlier tasks.](assets/dl22-continual-learning.svg){fig-align="center" width="74%" fig-alt="Tasks arrive sequentially, evaluation covers all tasks, and a feedback arc shows replay or regularization supporting retention."}

<details>
<summary><strong>PyTorch: measure forgetting and replay on sequential digit tasks</strong></summary>

```python
task_a_train = train_ids[np.isin(digits.target[train_ids], [0, 1, 2, 3, 4])]
task_b_train = train_ids[np.isin(digits.target[train_ids], [5, 6, 7, 8, 9])]
task_a_test = test_ids[np.isin(digits.target[test_ids], [0, 1, 2, 3, 4])]
task_b_test = test_ids[np.isin(digits.target[test_ids], [5, 6, 7, 8, 9])]


def continue_training(model, ids, seed, epochs=10, replay_ids=None):
    if replay_ids is not None:
        ids = np.concatenate([ids, replay_ids])
    optimizer = torch.optim.SGD(model.parameters(), lr=0.06, momentum=0.8)
    for epoch in range(epochs):
        model.train()
        for x, y in make_loader(ids, shuffle=True, seed=seed + epoch):
            loss = F.cross_entropy(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return model


seed_everything(2240)
naive_stream = DigitMLP(width=64, dropout=0.0)
continue_training(naive_stream, task_a_train, seed=2240, epochs=12)
a_before = evaluate(naive_stream, task_a_test)["accuracy"]
continue_training(naive_stream, task_b_train, seed=2250, epochs=12)
a_after = evaluate(naive_stream, task_a_test)["accuracy"]
b_after = evaluate(naive_stream, task_b_test)["accuracy"]

seed_everything(2240)
replay_stream = DigitMLP(width=64, dropout=0.0)
continue_training(replay_stream, task_a_train, seed=2240, epochs=12)
rng = np.random.default_rng(2240)
replay_ids = rng.choice(task_a_train, size=160, replace=False)
continue_training(
    replay_stream, task_b_train, seed=2250, epochs=12, replay_ids=replay_ids
)
replay_a = evaluate(replay_stream, task_a_test)["accuracy"]
replay_b = evaluate(replay_stream, task_b_test)["accuracy"]

assert set(targets[task_a_train].tolist()).isdisjoint(set(targets[task_b_train].tolist()))
print({
    "naive": {"A_before": round(a_before, 3), "A_after": round(a_after, 3), "B_after": round(b_after, 3)},
    "replay": {"A_after": round(replay_a, 3), "B_after": round(replay_b, 3)},
})
```

</details>

Replay is often effective, but it stores data and changes the sampling distribution. Other families constrain important parameters, isolate task-specific capacity, distill old behavior, or expand the model. Evaluation should report average accuracy across learned tasks, forgetting, forward transfer, memory, update compute, and whether task boundaries or labels are supplied. A method that retains everything by storing all data and retraining from scratch solves a different resource problem.


### **Causal Representation Learning** {#causal-representation-learning}

Standard predictive learning estimates associations that are useful under the training distribution. Causal representation learning asks whether high-dimensional observations can be mapped to variables whose relationships remain meaningful under interventions and environmental changes. In a structural causal model,

$$
z_i=f_i(\operatorname{pa}(z_i),\epsilon_i),
$$

each latent variable (z_i) is generated from its parents and independent disturbance (epsilon_i). Learning such variables from pixels or text is difficult because many latent descriptions fit the same observational distribution; identifiability requires assumptions, multiple environments, interventions, temporal structure, or other supervision.

![A stable causal feature and an environment-dependent shortcut can both predict the label during training.](assets/dl22-causal-shift.svg){fig-align="center" width="72%" fig-alt="A causal diagram shows environment affecting both digit content and a shortcut, with both feeding prediction."}

[Scholkopf et al.](https://arxiv.org/abs/2102.11107) connect causal representations to transfer and generalization: mechanisms may remain stable while nuisance associations change. This does not mean invariance alone identifies causality. A constant but wrong feature can appear invariant over the environments a researcher happened to collect.

<details>
<summary><strong>Python: construct and diagnose a spurious training environment</strong></summary>

```python
rng = np.random.default_rng(2260)
train_parity = digits.target[train_ids] % 2
test_parity = digits.target[test_ids] % 2

# The shortcut agrees with parity on 95% of training examples and is reversed
# at test time. Pixels remain the stable evidence source.
train_shortcut = np.where(rng.random(len(train_ids)) < 0.95, train_parity, 1 - train_parity)
test_shortcut = 1 - test_parity
stable_train = digits.data[train_ids] / 16.0
stable_test = digits.data[test_ids] / 16.0
spurious_train = np.column_stack([stable_train, 6.0 * train_shortcut])
spurious_test = np.column_stack([stable_test, 6.0 * test_shortcut])

stable_classifier = LogisticRegression(max_iter=500, random_state=2260).fit(
    stable_train, train_parity
)
spurious_classifier = LogisticRegression(max_iter=500, random_state=2260).fit(
    spurious_train, train_parity
)
stable_accuracy = accuracy_score(test_parity, stable_classifier.predict(stable_test))
shifted_accuracy = accuracy_score(test_parity, spurious_classifier.predict(spurious_test))
assert np.mean(train_shortcut == train_parity) > 0.90
assert np.mean(test_shortcut == test_parity) == 0.0
print({"stable_pixels": round(stable_accuracy, 3), "with_reversed_shortcut": round(shifted_accuracy, 3)})
```

</details>

The intervention is synthetic and the pixel model is not proven causal. Its purpose is to expose a falsifiable failure mode: a predictor selected because it was highly reliable in one environment can collapse when the data-collection mechanism changes. Stronger evidence would collect multiple real environments, define plausible interventions, test counterfactual predictions where identifiable, and state which causal assumptions cannot be verified from the data.


### **Neuro-Symbolic Learning** {#neuro-symbolic-learning}

Neuro-symbolic systems combine learned perception or representation with explicit structure such as logical rules, programs, knowledge graphs, types, or constraints. Integration can occur at several points: symbolic knowledge may generate supervision, constrain the loss, guide search, filter decoded outputs, or operate as a separate reasoning module after neural perception.

![Neural perception produces uncertain scores while a symbolic layer enforces explicit validity constraints.](assets/dl22-neuro-symbolic-loop.svg){fig-align="center" width="74%" fig-alt="Raw input passes through a neural model and a symbolic constraint layer, with a feedback path from constraints to learning."}

The attraction is complementary strength. Neural models tolerate noise and high-dimensional inputs; symbolic systems can express compositional rules and produce inspectable proof steps. The interface is also the main weakness: discrete decisions can block gradients, neural uncertainty may be discarded too early, rule bases may be incomplete, and a wrong hard constraint can force a confidently wrong output.

<details>
<summary><strong>PyTorch: use a parity constraint during digit-pair decoding</strong></summary>

```python
baseline_model.eval()
pair_ids = test_ids[:200]
pair_x = images[pair_ids]
pair_y = targets[pair_ids]
generator = torch.Generator().manual_seed(2270)
pair_x = (pair_x + 0.28 * torch.randn(pair_x.shape, generator=generator)).clamp(0, 1)
with torch.inference_mode():
    pair_probabilities = baseline_model(pair_x).softmax(dim=1)

direct_correct = 0
constrained_correct = 0
pair_count = len(pair_ids) // 2
for i in range(pair_count):
    p1, p2 = pair_probabilities[2 * i], pair_probabilities[2 * i + 1]
    y1, y2 = int(pair_y[2 * i]), int(pair_y[2 * i + 1])
    known_parity = (y1 + y2) % 2  # External symbolic side information.
    direct = (int(p1.argmax()), int(p2.argmax()))
    direct_correct += int(direct == (y1, y2))

    best_score, best_pair = -1.0, None
    for a in p1.topk(4).indices.tolist():
        for b in p2.topk(4).indices.tolist():
            if (a + b) % 2 == known_parity:
                score = float(p1[a] * p2[b])
                if score > best_score:
                    best_score, best_pair = score, (a, b)
    constrained_correct += int(best_pair == (y1, y2))

assert constrained_correct >= direct_correct
print({"pairs": pair_count, "direct_correct": direct_correct, "constrained_correct": constrained_correct})
```

</details>

The constraint cannot damage an already correct pair because the true pair satisfies it, but it uses privileged parity information unavailable in ordinary recognition. The comparison therefore demonstrates constrained decoding, not a free accuracy improvement. Research must price acquisition of symbolic knowledge, test rule violations and contradictions, preserve uncertainty across the interface, and compare against neural models given equivalent side information.


### **Choosing an Open Research Problem** {#choosing-open-research-problem}

An open topic is not yet a research problem. “Work on world models” names an area; “determine whether uncertainty-gated planning reduces unsafe model exploitation under a fixed interaction budget” proposes a falsifiable relationship. A good problem connects importance, an unresolved mechanism, feasible evidence, and a decision that can change after the experiment.

![Successive constraints turn a broad important area into a researchable decision.](assets/dl22-problem-funnel.svg){fig-align="center" width="68%" fig-alt="A four-stage funnel narrows from an important problem to a falsifiable question, feasible evidence, and a decision rule."}

Use five filters:

1. **Importance:** who or what changes if the question is answered, and what is the cost of a wrong answer?
2. **Gap:** is the uncertainty scientific, empirical, engineering, or evaluative? A missing benchmark entry is not automatically a meaningful gap.
3. **Tractability:** can the decisive experiment be run with available data, compute, expertise, and time?
4. **Identifiability:** could the proposed evidence distinguish the mechanism from confounders and competing explanations?
5. **Contribution path:** would a negative result still produce a useful dataset, protocol, diagnosis, bound, or design rule?

<details>
<summary><strong>Python: make problem selection criteria explicit</strong></summary>

```python
candidates = {
    "adaptive_test_time_compute": {
        "importance": 4, "gap": 4, "tractability": 5, "identifiability": 4, "negative_value": 5
    },
    "moe_routing_balance": {
        "importance": 4, "gap": 3, "tractability": 4, "identifiability": 4, "negative_value": 4
    },
    "causal_digit_representation": {
        "importance": 5, "gap": 5, "tractability": 2, "identifiability": 1, "negative_value": 3
    },
}
weights = {
    "importance": 0.25,
    "gap": 0.20,
    "tractability": 0.20,
    "identifiability": 0.25,
    "negative_value": 0.10,
}
scores = {
    name: sum(criteria[key] * weight for key, weight in weights.items())
    for name, criteria in candidates.items()
}
ranking = sorted(scores.items(), key=lambda item: item[1], reverse=True)
assert ranking[0][0] == "adaptive_test_time_compute"
print(ranking)
```

</details>

The numbers do not make the decision objective; they expose why it was made. Sensitivity analysis should vary the weights, and domain experts should challenge the importance and identifiability scores. A small, decisive experiment is usually a stronger starting point than a grand question whose only feasible test is an underpowered proxy.


### **Maintaining a Research Log** {#maintaining-research-log}

A research log records decisions while their alternatives are still visible. The unit is not “worked on model today,” but a reconstructable event: question, hypothesis, change from the previous run, configuration, data version, environment, result, interpretation, and next decision. Logging only successful runs creates hindsight bias and makes failed directions expensive to rediscover.

![A research log links questions, configurations, data, runs, metrics, decisions, and artifacts.](assets/dl22-research-log.svg){fig-align="center" width="76%" fig-alt="A provenance graph connects a research question to configuration, data, run, metrics, decision, and preserved artifact, with negative results feeding the next question."}

Separate immutable facts from interpretation. Configuration, checksums, raw predictions, timestamps, and environment metadata should be machine-generated. Hypotheses, surprises, and stop decisions require human explanation. Store the exact command and preserve generated artifacts under content hashes or versioned paths; a mutable file called `final_results.csv` is not provenance.

<details>
<summary><strong>Python: produce a content-addressed experiment record</strong></summary>

```python
run_config = {
    "model": "DigitMLP",
    "width": 64,
    "dropout": 0.10,
    "epochs": 12,
    "optimizer": "AdamW",
    "learning_rate": 2e-3,
    "seed": 2201,
}
config_json = json.dumps(run_config, sort_keys=True, separators=(",", ":"))
run_id = hashlib.sha256(config_json.encode("utf-8")).hexdigest()[:12]
research_record = {
    "run_id": run_id,
    "question": claim_card["question"],
    "config": run_config,
    "dataset_sha256": dataset_digest,
    "split_sha256": manifest["split_sha256"],
    "metrics": {k: round(v, 6) for k, v in baseline_metrics.items()},
    "interpretation": "Baseline for controlled width/dropout ablations.",
    "decision": "retain as reference; do not tune on test metrics",
}
record_json = json.dumps(research_record, indent=2, sort_keys=True)
assert research_record["run_id"] == hashlib.sha256(config_json.encode()).hexdigest()[:12]
assert research_record["dataset_sha256"] == manifest["dataset_sha256"]
print(record_json)
```

</details>

A mature project also records ethics and governance decisions, licenses, access restrictions, resource consumption, reviewer feedback, and deviations from the original plan. The [Turing Way](https://book.the-turing-way.org/) provides an open, CC BY 4.0 handbook for reproducible, ethical, and collaborative research. The point of the log is not bureaucracy; it is to preserve the causal history of the project well enough that another person, including the future author, can understand why an artifact exists.


### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Research practice and frontier awareness solve different problems. Practice determines whether evidence is trustworthy; frontier mapping determines where important uncertainty remains. A novel architecture studied with confounded baselines is weak research, while a perfectly reproducible experiment on an irrelevant question is merely well-organized. Strong work aligns both.

| Area | Core research question | Required evidence | Frequent failure |
|---|---|---|---|
| Reproduction and baselines | Can the result be regenerated and compared fairly? | artifacts, matched budgets, independent checks | treating one deterministic rerun as general replication |
| Ablation | Which intervention causes the change? | controlled factorial comparisons, multiple seeds | changing capacity, tuning, and data together |
| Scaling | How should finite resources be allocated? | multi-axis pilots, uncertainty, cost accounting | distant extrapolation from a narrow regime |
| Sparse and hybrid models | Can capacity grow without proportional active compute? | quality, routing, communication, latency, memory | equating theoretical FLOPs with system speed |
| Test-time compute | Which inputs benefit from more inference work? | matched cost curves and verifier analysis | ignoring selection and verification cost |
| World models | Can learned dynamics support safe planning? | intervention and long-horizon tests | judging usefulness by visual plausibility |
| Scientific learning | Does the model support a valid scientific workflow? | domain constraints, uncertainty, external validation | replacing scientific validity with benchmark error |
| Neural operators | Can a function-to-function map transfer across discretizations? | boundary, mesh, rollout, and solver comparisons | assuming FFT use guarantees physical fidelity |
| Continual learning | Can the system adapt while retaining prior capability? | forgetting, transfer, memory, update cost | hiding task labels or stored data assumptions |
| Causal representations | Which learned factors remain meaningful under intervention? | explicit assumptions, environments, interventions | calling correlation or invariance causal |
| Neuro-symbolic learning | How should learned uncertainty interact with explicit rules? | interface tests, rule failures, equivalent side information | treating privileged constraints as free accuracy |

The chapter's experiments form one compact evidence trail: a fixed UCI split, a declared question, repeatable baseline, factorial ablation, scaling pilot, sparse and routed variants, a test-time compute curve, sequential tasks, a controlled shortcut shift, constrained decoding, and a content-addressed log. Each result is deliberately scoped as a mechanism study. That discipline is the transferable skill: ask what changed, what was held fixed, what evidence would reverse the conclusion, and what the current experiment cannot establish.

Emerging frontiers should be revisited as evidence changes. Sparse experts, hybrid sequence models, test-time computation, world models, scientific learning, neural operators, continual adaptation, causal representation, and neuro-symbolic integration are not a single roadmap to “general intelligence.” They are distinct responses to limits in capacity allocation, inference, interaction, scientific structure, adaptation, and reasoning. The most durable research contribution may be a model, but it may equally be a sharper question, a reliable evaluation protocol, a negative result, a dataset with better provenance, or a boundary that prevents an attractive claim from traveling too far.
